# Gaussian: can RobustCE beat ordinary CE on clean test CE?

## 1. Question and comparison rule

Sweep training-label flip probability **eta** and RobustCE training radius **delta** for the same Gaussian logistic-regression problem as notebook 02.

At every eta, compare RobustCE against an ordinary-CE model trained with the **same corrupted labels, data, initialization, optimizer, and epoch budget**. Tune learning rate separately for the CE baseline and every RobustCE radius using clean validation CE and three tuning seeds. Then freeze those rates and evaluate on five new matched seeds.

The primary score is
$$
\mathrm{gain}(\eta,\delta)
=\frac{1}{S}\sum_s\left[
\mathrm{CE}_{\mathrm{test}}(\mathrm{CE},\eta,s)
-\mathrm{CE}_{\mathrm{test}}(\mathrm{RobustCE},\eta,\delta,s)
\right].
$$
**Positive gain means RobustCE has lower ordinary clean test CE.**

The heatmap uses a continuous red-white-green scale centered at zero: red for worse clean test CE, green for better CE, and white for zero or negligible change. Saturation represents absolute gain magnitude. Test outcomes color the exploratory grid; they are not used to choose a deployable radius. Validation and test labels always remain clean.

All text is in English, all plots use Plotly, and no tables are displayed. The notebook runs independently with uv and PyTorch on CPU.


In [1]:
import csv
import hashlib
import json
import statistics
import time
from itertools import product
from pathlib import Path

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
RESULTS_ROOT = PROJECT_ROOT / "results" / "03_gaussian_eta_delta"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

ETAS = (0.0, 0.025, 0.05, 0.1, 0.2, 0.3, 0.4)
DELTAS = (0.0, 0.0001, 0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1, 0.2, 0.4, 0.5)
LEARNING_RATES = (0.001, 0.01, 0.05)
TUNING_SEEDS = (0, 1, 2)
EVALUATION_SEEDS = (201, 202, 203, 204, 205)
SPLIT_SIZES = {"train": 1000, "validation": 400, "test": 2000}
EPOCHS = 400
WIN_ATOL = 1e-6
REUSE_CACHE = True

assert set(TUNING_SEEDS).isdisjoint(EVALUATION_SEEDS)
assert ETAS == tuple(sorted(set(ETAS))) and DELTAS == tuple(sorted(set(DELTAS)))
assert all(0 <= eta < 0.5 for eta in ETAS)
assert all(0 <= delta <= 1 for delta in DELTAS) and 0.0 in DELTAS
assert all(size % 2 == 0 for size in SPLIT_SIZES.values())
TOTAL_TUNING_RUNS = len(ETAS) * len(DELTAS) * len(LEARNING_RATES) * len(TUNING_SEEDS)
TOTAL_EVALUATION_RUNS = len(ETAS) * len(DELTAS) * len(EVALUATION_SEEDS)

print(f"PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")
print(f"Grid: {len(ETAS)} eta values x {len(DELTAS)} delta values.")
print(f"Tuning: {TOTAL_TUNING_RUNS} unique runs; evaluation: {TOTAL_EVALUATION_RUNS} unique runs.")
print(f"Win: mean paired clean-test CE gain > {WIN_ATOL:g}.")


PyTorch 2.11.0+cpu | cpu | torch.float32
Grid: 7 eta values x 12 delta values.
Tuning: 756 unique runs; evaluation: 420 unique runs.
Win: mean paired clean-test CE gain > 1e-06.


## 2. Matched data and training

Generate balanced classes with
$$
x\mid y\sim\mathcal N((2y-1)(1,1)^\top,I_2),\qquad y\in\{0,1\}.
$$
Independently generate train, validation, and test splits. Standardize features using only training-feature statistics.

Each seed varies the dataset and model initialization. Within a seed, reuse both across all eta/delta/lr combinations. Draw one uniform vector per seed and flip training label $i$ iff $u_i<\eta$; this shares the exact corruption masks across methods and makes masks nested across eta.

Use the same 3-parameter logistic model and 400 full-batch Adam steps for all runs. Delta = 0 is the ordinary-CE baseline and is trained only once per eta/seed/lr combination, not once for every positive delta.


In [2]:
def generator(seed):
    return torch.Generator(device="cpu").manual_seed(seed)


def gaussian_split(n, seed):
    g = generator(seed)
    labels = torch.cat((torch.zeros(n // 2), torch.ones(n // 2))).to(DTYPE)
    features = torch.randn(n, 2, generator=g, dtype=DTYPE) + (2 * labels - 1).unsqueeze(1)
    permutation = torch.randperm(n, generator=g)
    return features[permutation].to(DEVICE), labels[permutation].to(DEVICE)


def prepare_data(seed, include_test=False):
    splits = ("train", "validation", "test") if include_test else ("train", "validation")
    data = {
        split: gaussian_split(SPLIT_SIZES[split], seed * 1000 + split_id)
        for split_id, split in enumerate(splits)
    }
    mean = data["train"][0].mean(dim=0)
    scale = data["train"][0].std(dim=0, unbiased=False)
    assert torch.all(scale > 0)
    for split, (features, labels) in data.items():
        data[split] = ((features - mean) / scale, labels)
    data["corruption_uniforms"] = torch.rand(
        SPLIT_SIZES["train"], generator=generator(seed * 1000 + 999), dtype=DTYPE
    )
    return data


def observed_labels(data, eta):
    clean = data["train"][1]
    observed = torch.where(data["corruption_uniforms"] < eta, 1 - clean, clean)
    realized = (observed != clean).to(DTYPE).mean().item()
    return observed, realized


def make_model(seed):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed + 10_000)
        model = nn.Linear(2, 1)
    return model.to(device=DEVICE, dtype=DTYPE)


assert sum(p.numel() for p in make_model(0).parameters()) == 3


In [3]:
def objective(logits, labels, delta):
    ce = F.softplus(logits) - labels * logits
    if delta == 0:
        return ce.mean()
    return (ce + min(1.0, delta) * F.relu((2 * labels - 1) * logits)).mean()


@torch.no_grad()
def evaluate(model, split):
    features, labels = split
    logits = model(features).squeeze(-1)
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite predictions.")
    return {
        "ce": F.binary_cross_entropy_with_logits(logits, labels).item(),
        "accuracy": ((logits >= 0) == labels.bool()).to(DTYPE).mean().item(),
        "confidence": torch.sigmoid(logits.abs()).mean().item(),
    }


def train_one(seed, data, observed, delta, lr):
    model = make_model(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    features = data["train"][0]
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        optimizer.zero_grad(set_to_none=True)
        loss = objective(model(features).squeeze(-1), observed, delta)
        if epoch % 20 == 0 and not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}.")
        loss.backward()
        optimizer.step()
    seconds = time.perf_counter() - started
    model.eval()
    return model, evaluate(model, data["validation"]), seconds


def summarize(values):
    return statistics.mean(values), statistics.stdev(values) if len(values) > 1 else 0.0


def write_csv(path, records):
    if not records:
        return
    fields = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def read_csv(path):
    records = []
    with path.open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            parsed = {}
            for key, value in row.items():
                if key in ("status", "error"):
                    parsed[key] = value
                elif value == "":
                    parsed[key] = None
                elif key == "seed":
                    parsed[key] = int(value)
                else:
                    parsed[key] = float(value)
            records.append(parsed)
    return records


def run_key(record):
    return record["seed"], record["eta"], record["delta"], record["lr"]


### Reproducibility and resumable runs

Cache successful run records after each seed/eta block. A fingerprint includes the numerical settings, PyTorch version, and the exact data/training function sources. Changing those settings or functions creates a separate result directory; plot-only edits can reuse completed runs.

Failed runs are recorded and are retried on a later execution. A tuning candidate with any failed/missing seed is ineligible. A heatmap cell without all five matched evaluation seeds is shown as missing rather than counted as a win.


In [4]:
settings = {
    "torch_version": str(torch.__version__),
    "device": str(DEVICE), "dtype": str(DTYPE), "threads": torch.get_num_threads(),
    "etas": list(ETAS), "deltas": list(DELTAS), "learning_rates": list(LEARNING_RATES),
    "tuning_seeds": list(TUNING_SEEDS), "evaluation_seeds": list(EVALUATION_SEEDS),
    "split_sizes": SPLIT_SIZES, "epochs": EPOCHS, "win_atol": WIN_ATOL,
    "optimizer": "Adam, default betas/eps, no weight decay, no schedule",
    "batching": "full batch", "selection": "mean final ordinary clean validation CE",
}
notebook_path = PROJECT_ROOT / "notebooks" / "03_gaussian_eta_delta.ipynb"
notebook_source = json.loads(notebook_path.read_text(encoding="utf-8"))
implementation = "\n".join(
    "".join(cell["source"]) for cell in notebook_source["cells"]
    if cell.get("id") in ("data", "training")
)
fingerprint = hashlib.sha256(
    json.dumps({"settings": settings, "implementation": implementation}, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
RUN_DIR = RESULTS_ROOT / fingerprint
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
(RESULTS_ROOT / "latest_run.json").write_text(
    json.dumps({"run_id": fingerprint, "run_directory": str(RUN_DIR)}, indent=2) + "\n", encoding="utf-8"
)
TUNING_PATH = RUN_DIR / "tuning_runs.csv"
EVALUATION_PATH = RUN_DIR / "evaluation_runs.csv"
tuning_cache = {
    run_key(row): row for row in (read_csv(TUNING_PATH) if REUSE_CACHE and TUNING_PATH.exists() else [])
}
evaluation_cache = {
    run_key(row): row for row in (read_csv(EVALUATION_PATH) if REUSE_CACHE and EVALUATION_PATH.exists() else [])
}
print(f"Result directory: {RUN_DIR}")
print(f"Cached records: {len(tuning_cache)} tuning, {len(evaluation_cache)} evaluation.")


Result directory: C:\Users\shich\Src\hw1opt\results\03_gaussian_eta_delta\f561c0d38c3a3d19
Cached records: 756 tuning, 420 evaluation.


## 3. Validation-only learning-rate selection

For every eta and delta, search the same three learning rates using three tuning seeds. Corrupt only training labels. Validation labels are clean.

Select each configuration's rate by mean final validation CE. In particular, ordinary CE gets its own rate at every eta, making it the baseline for all deltas at that eta. No test split is generated during tuning.


In [5]:
started = time.perf_counter()
new_tuning_runs = 0
tuning_data = {seed: prepare_data(seed, include_test=False) for seed in TUNING_SEEDS}

for seed, eta in product(TUNING_SEEDS, ETAS):
    data = tuning_data[seed]
    observed, realized = observed_labels(data, eta)
    for delta, lr in product(DELTAS, LEARNING_RATES):
        record = dict(seed=seed, eta=eta, delta=delta, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in tuning_cache and tuning_cache[key]["status"] == "ok":
            continue
        try:
            _, validation, seconds = train_one(seed, data, observed, delta, lr)
            record.update(
                status="ok", validation_ce=validation["ce"],
                validation_accuracy=validation["accuracy"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        tuning_cache[key] = record
        new_tuning_runs += 1
    write_csv(TUNING_PATH, list(tuning_cache.values()))
    print(f"Tuning seed={seed}, eta={eta:g}: {len(tuning_cache)}/{TOTAL_TUNING_RUNS} records.", flush=True)

tuning_records = list(tuning_cache.values())
tuning_summary = []
selected = {}
for eta, delta in product(ETAS, DELTAS):
    candidates = []
    for lr in LEARNING_RATES:
        runs = [tuning_cache.get((seed, eta, delta, lr)) for seed in TUNING_SEEDS]
        eligible = all(row is not None and row["status"] == "ok" for row in runs)
        entry = dict(eta=eta, delta=delta, lr=lr, eligible=eligible)
        if eligible:
            entry["validation_ce_mean"], entry["validation_ce_std"] = summarize([r["validation_ce"] for r in runs])
            candidates.append(entry)
        tuning_summary.append(entry)
    if not candidates:
        raise RuntimeError(f"No complete valid learning-rate candidate for eta={eta}, delta={delta}.")
    best = min(candidates, key=lambda row: (row["validation_ce_mean"], row["lr"]))
    selected[(eta, delta)] = best["lr"]

frozen_settings = [
    dict(eta=eta, delta=delta, lr=lr) for (eta, delta), lr in selected.items()
]
selection_snapshot = json.dumps(frozen_settings, sort_keys=True)
(RUN_DIR / "selected_settings.json").write_text(json.dumps(frozen_settings, indent=2) + "\n", encoding="utf-8")
write_csv(RUN_DIR / "tuning_summary.csv", tuning_summary)
boundary_count = sum(lr in (min(LEARNING_RATES), max(LEARNING_RATES)) for lr in selected.values())
print(f"Tuning finished in {time.perf_counter()-started:.1f} s; {new_tuning_runs} newly trained runs.")
print(f"Selected rates at tested grid endpoints: {boundary_count}/{len(selected)}. Values outside this finite grid were not tested.")


Tuning seed=0, eta=0: 756/756 records.


Tuning seed=0, eta=0.025: 756/756 records.


Tuning seed=0, eta=0.05: 756/756 records.


Tuning seed=0, eta=0.1: 756/756 records.


Tuning seed=0, eta=0.2: 756/756 records.


Tuning seed=0, eta=0.3: 756/756 records.


... (13 outputs left out)

Tuning seed=2, eta=0.3: 756/756 records.

Tuning seed=2, eta=0.4: 756/756 records.


Tuning finished in 0.2 s; 0 newly trained runs.


Selected rates at tested grid endpoints: 78/84. Values outside this finite grid were not tested.


## 4. Evaluation on new matched seeds

Use five new seeds, 201 through 205. Generate clean validation/test labels and reuse each seed's data, initialization, and corruption uniforms across all cells.

Train each model from scratch using its frozen learning rate. Delta = 0 is the ordinary-CE baseline; it is reused across columns when computing paired gains. The final checkpoint is evaluated once on clean test data. No checkpoint or radius is selected by test loss.


In [6]:
started = time.perf_counter()
new_evaluation_runs = 0
evaluation_data = {seed: prepare_data(seed, include_test=True) for seed in EVALUATION_SEEDS}

for seed, eta in product(EVALUATION_SEEDS, ETAS):
    data = evaluation_data[seed]
    observed, realized = observed_labels(data, eta)
    for delta in DELTAS:
        lr = selected[(eta, delta)]
        record = dict(seed=seed, eta=eta, delta=delta, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in evaluation_cache and evaluation_cache[key]["status"] == "ok":
            continue
        try:
            model, validation, seconds = train_one(seed, data, observed, delta, lr)
            test = evaluate(model, data["test"])
            record.update(
                status="ok", validation_ce=validation["ce"],
                test_ce=test["ce"], test_accuracy=test["accuracy"],
                test_confidence=test["confidence"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        evaluation_cache[key] = record
        new_evaluation_runs += 1
    write_csv(EVALUATION_PATH, list(evaluation_cache.values()))
    print(f"Evaluation seed={seed}, eta={eta:g}: {len(evaluation_cache)}/{TOTAL_EVALUATION_RUNS} records.", flush=True)

evaluation_records = list(evaluation_cache.values())
assert selection_snapshot == json.dumps([
    dict(eta=eta, delta=delta, lr=lr) for (eta, delta), lr in selected.items()
], sort_keys=True)
print(f"Evaluation finished in {time.perf_counter()-started:.1f} s; {new_evaluation_runs} newly trained runs.")


Evaluation seed=201, eta=0: 420/420 records.


Evaluation seed=201, eta=0.025: 420/420 records.

Evaluation seed=201, eta=0.05: 420/420 records.

Evaluation seed=201, eta=0.1: 420/420 records.


Evaluation seed=201, eta=0.2: 420/420 records.


Evaluation seed=201, eta=0.3: 420/420 records.


... (26 outputs left out)

Evaluation seed=205, eta=0.2: 420/420 records.


Evaluation seed=205, eta=0.3: 420/420 records.

Evaluation seed=205, eta=0.4: 420/420 records.


Evaluation finished in 0.3 s; 0 newly trained runs.


## 5. Paired comparison against ordinary CE

A cell is a mean across the same five paired evaluation seeds. Save both losses, mean gain, relative reduction, sample standard deviation of paired gains, and the number of seeds with gain above the numerical tolerance.

A positive five-seed mean is an observed win in this experiment, not a claim of statistical significance. The delta = 0 control column must have exactly zero gain.


In [7]:
paired_records = []
cell_records = []
for eta, delta in product(ETAS, DELTAS):
    baseline_runs, robust_runs, gains = [], [], []
    for seed in EVALUATION_SEEDS:
        baseline = evaluation_cache.get((seed, eta, 0.0, selected[(eta, 0.0)]))
        robust = evaluation_cache.get((seed, eta, delta, selected[(eta, delta)]))
        if baseline is None or robust is None or baseline["status"] != "ok" or robust["status"] != "ok":
            continue
        assert baseline["realized_corruption"] == robust["realized_corruption"]
        gain = baseline["test_ce"] - robust["test_ce"]
        baseline_runs.append(baseline)
        robust_runs.append(robust)
        gains.append(gain)
        paired_records.append(dict(
            seed=seed, eta=eta, delta=delta,
            baseline_lr=baseline["lr"], robust_lr=robust["lr"],
            baseline_test_ce=baseline["test_ce"], robust_test_ce=robust["test_ce"],
            gain=gain, win=gain > WIN_ATOL,
        ))
    complete = len(gains) == len(EVALUATION_SEEDS)
    record = dict(
        eta=eta, delta=delta, complete=complete, paired_seeds=len(gains),
        baseline_lr=selected[(eta, 0.0)], robust_lr=selected[(eta, delta)],
    )
    if complete:
        baseline_mean = statistics.mean(r["test_ce"] for r in baseline_runs)
        robust_mean = statistics.mean(r["test_ce"] for r in robust_runs)
        gain_mean, gain_std = summarize(gains)
        record.update(
            baseline_test_ce_mean=baseline_mean, robust_test_ce_mean=robust_mean,
            gain_mean=gain_mean, gain_std=gain_std,
            relative_gain_percent=100 * gain_mean / baseline_mean,
            seed_wins=sum(gain > WIN_ATOL for gain in gains),
            mean_win=gain_mean > WIN_ATOL,
        )
        if delta == 0:
            assert all(gain == 0 for gain in gains)
    cell_records.append(record)

write_csv(RUN_DIR / "paired_results.csv", paired_records)
write_csv(RUN_DIR / "grid_summary.csv", cell_records)
complete_cells = [r for r in cell_records if r["complete"]]
nonzero_cells = [r for r in complete_cells if r["delta"] > 0]
winning_cells = [r for r in nonzero_cells if r["mean_win"]]
print(f"Complete cells: {len(complete_cells)}/{len(ETAS)*len(DELTAS)}.")
print(f"Observed mean wins at delta > 0: {len(winning_cells)}/{len(nonzero_cells)}.")
if winning_cells:
    for eta in ETAS:
        wins = [r for r in winning_cells if r["eta"] == eta]
        if wins:
            print(f"eta={eta:g}: {len(wins)} winning cells; deltas=" + ", ".join(f"{r['delta']:g}" for r in wins))
else:
    print("No tested positive delta beat its matched CE baseline beyond the numerical tolerance.")


Complete cells: 84/84.


Observed mean wins at delta > 0: 3/77.
eta=0: 3 winning cells; deltas=0.0001, 0.001, 0.0025


## 6. Eta versus delta: red-white-green comparison grid

- **Green:** positive gain; RobustCE lowers ordinary clean test CE.
- **Red:** negative gain; RobustCE increases ordinary clean test CE.
- **White:** zero or very small absolute change. Numerical ties are mapped exactly to zero.
- **Saturation:** absolute gain magnitude on one symmetric linear scale centered at zero.
- **Missing:** a cell without all five valid paired seeds is left blank.

Every tested pair occupies an equal-size categorical bin. Cell labels and hover preserve the actual signed mean gain; hover also shows both losses, relative change, paired variation, seed wins, and selected learning rates.

The numerical win rule remains mean gain > 0.000001. Very small observed wins can look almost white because they are small on the common loss scale. The zero-delta control is white.


In [8]:
by_cell = {(r["eta"], r["delta"]): r for r in cell_records}
color_grid, text_grid, hover_grid = [], [], []
for eta in ETAS:
    colors, texts, details = [], [], []
    for delta in DELTAS:
        record = by_cell[(eta, delta)]
        if not record["complete"]:
            colors.append(None)
            texts.append("missing")
            details.append([eta, delta, None, None, None, None, None, None, None, None, None])
            continue
        gain = record["gain_mean"]
        # Keep zero and numerical ties white; use actual signed gains elsewhere.
        color_gain = 0.0 if abs(gain) <= WIN_ATOL else gain
        colors.append(color_gain)
        texts.append("tie" if abs(gain) <= WIN_ATOL else f"{gain:+.3f}" if abs(gain) >= 0.001 else f"{gain:+.1e}")
        details.append([
            eta, delta, record["baseline_test_ce_mean"], record["robust_test_ce_mean"],
            gain, record["relative_gain_percent"], record["gain_std"],
            record["seed_wins"], record["paired_seeds"], record["baseline_lr"], record["robust_lr"],
        ])
    color_grid.append(colors)
    text_grid.append(texts)
    hover_grid.append(details)

color_limit = max([abs(v) for row in color_grid for v in row if v is not None] + [WIN_ATOL])
colorscale = [
    [0.0, "#991b1b"],
    [0.5, "#ffffff"],
    [1.0, "#166534"],
]
fig = go.Figure(go.Heatmap(
    x=[f"{delta:g}" for delta in DELTAS],
    y=[f"{eta:g}" for eta in ETAS],
    z=color_grid, zmin=-color_limit, zmax=color_limit,
    colorscale=colorscale,
    text=text_grid, texttemplate="%{text}", textfont=dict(size=10, color="#111827"),
    customdata=hover_grid, xgap=2, ygap=2, hoverongaps=False,
    colorbar=dict(title="CE baseline - RobustCE", thickness=18),
    hovertemplate=(
        "eta=%{customdata[0]}<br>delta=%{customdata[1]}"
        "<br>Mean baseline test CE=%{customdata[2]:.8f}"
        "<br>Mean RobustCE test CE=%{customdata[3]:.8f}"
        "<br>Mean paired gain=%{customdata[4]:+.8f}"
        "<br>Relative CE reduction=%{customdata[5]:+.4f}%"
        "<br>Paired gain sample SD=%{customdata[6]:.8f}"
        "<br>Seed wins=%{customdata[7]}/%{customdata[8]}"
        "<br>Selected baseline lr=%{customdata[9]}"
        "<br>Selected RobustCE lr=%{customdata[10]}"
        "<extra></extra>"
    ),
))
fig.update_xaxes(title_text="RobustCE training radius delta", type="category", tickangle=-35, side="bottom")
fig.update_yaxes(title_text="Training-label flip probability eta", type="category", autorange="reversed")
fig.update_layout(
    title="Gaussian: did RobustCE beat ordinary CE on clean test CE?",
    height=650, margin=dict(l=90, r=180, t=100, b=120),
    plot_bgcolor="#e5e7eb",
    annotations=[dict(
        x=0, y=1.08, xref="paper", yref="paper", showarrow=False, xanchor="left",
        text="Green: lower CE; red: higher CE; white: near zero. Five matched evaluation seeds.",
    )],
)
fig.show(config={"displaylogo": False, "responsive": True})
fig.write_html(RUN_DIR / "eta_delta_heatmap.html", include_plotlyjs=True, full_html=True)


## 7. Interpretation and scope

Green cells identify observed reductions in ordinary clean test CE relative to a separately validation-tuned CE baseline at the same training-label corruption level. Red cells mean worse CE; white or nearly white cells indicate zero or small changes. Finding no meaningful improvement is a valid result.

Read magnitude and repeatability together: hover provides the mean gain, its paired sample standard deviation, and wins among five seeds. A tiny green gain may be practically negligible or sensitive to seed choice. Searching many test cells is exploratory; a candidate configuration should be frozen and checked on fresh seeds before claiming a reproducible advantage.

This experiment keeps the correctly specified 2D Gaussian logistic model, 1,000 training examples, a finite learning-rate grid, and 400 optimization steps. It does not imply that RobustCE can never improve test CE in other sample-size, model-capacity, optimization, or corruption settings.


In [9]:
failures = [
    {"phase": phase, **record}
    for phase, records in (("tuning", tuning_records), ("evaluation", evaluation_records))
    for record in records if record["status"] != "ok"
]
protocol = {
    **settings, "run_id": fingerprint,
    "test_metric": "ordinary CE on clean held-out labels",
    "gain": "matched CE baseline test CE minus RobustCE test CE",
    "baseline": "delta=0, independently validation-tuned lr at each eta",
    "win_rule": f"five-seed mean paired gain > {WIN_ATOL}",
    "heatmap_colors": "continuous red-white-green; zero-centered; numerical ties at zero",
    "zero_radius_control": "exact same CE model/record reused; gain must be zero",
    "cache_reuse": REUSE_CACHE,
    "unique_tuning_runs": TOTAL_TUNING_RUNS,
    "unique_evaluation_runs": TOTAL_EVALUATION_RUNS,
    "complete_cells": len(complete_cells),
    "positive_delta_cells": len(nonzero_cells),
    "observed_winning_cells": len(winning_cells),
    "failed_runs": len(failures),
    "scope": "exploratory eta/delta test grid; no test-based deployment selection",
}
(RUN_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
(RUN_DIR / "failures.json").write_text(json.dumps(failures, indent=2) + "\n", encoding="utf-8")
assert len(tuning_records) == TOTAL_TUNING_RUNS
assert len(evaluation_records) == TOTAL_EVALUATION_RUNS
assert len(cell_records) == len(ETAS) * len(DELTAS)
assert len([r for r in evaluation_records if r["delta"] == 0]) == len(ETAS) * len(EVALUATION_SEEDS)
print(
    f"Saved {len(cell_records)} cells from {len(tuning_records)} tuning and "
    f"{len(evaluation_records)} evaluation records. Failed runs: {len(failures)}."
)
print(f"Artifacts and interactive HTML: {RUN_DIR}")


Saved 84 cells from 756 tuning and 420 evaluation records. Failed runs: 0.
Artifacts and interactive HTML: C:\Users\shich\Src\hw1opt\results\03_gaussian_eta_delta\f561c0d38c3a3d19
